In [1]:
!module load python

In [2]:
import os
import torch
from pathlib import Path
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers import AutoTokenizer, AutoModelForCausalLM

is_cuda = torch.cuda.is_available()
num = torch.cuda.device_count()
name = torch.cuda.get_device_name(0) if is_cuda and num > 0 else "None"

print(f"CUDA available: {is_cuda}")
print(f"GPU count: {num}")
print(f"Primary GPU: {name}")

device = "cuda"


hub = Path(os.path.expanduser("~/.cache/huggingface/hub"))
base = hub / "models--codellama--CodeLlama-7b-Instruct-hf"

# Prefer the ref in refs/main; fall back to the newest snapshot
ref_file = base / "refs" / "main"
commit = ref_file.read_text().strip()

MODEL_PATH = str(base / "snapshots" / commit)
print("MODEL_PATH =", MODEL_PATH)

# tokenizer
tok = AutoTokenizer.from_pretrained(
    MODEL_PATH,
    use_fast=True,
    local_files_only=True,
)

CUDA available: True
GPU count: 1
Primary GPU: NVIDIA H100 80GB HBM3 MIG 3g.40gb
MODEL_PATH = /home/rpinter/.cache/huggingface/hub/models--codellama--CodeLlama-7b-Instruct-hf/snapshots/22cb240e0292b0b5ab4c17ccd97aa3a2f799cbed


---

In [6]:
model = AutoModelForCausalLM.from_pretrained(
        MODEL_PATH,
        local_files_only=True,
        device_map="auto",
        low_cpu_mem_usage=True,
    )

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [7]:
from transformers import TextStreamer
import torch

def ask(prompt, max_new_tokens=128, temperature=0.2, top_p=0.9):
    inst = f"<s>[INST] {prompt.strip()} [/INST]"
    inputs = tok(inst, return_tensors="pt").to(model.device)

    with torch.no_grad():
        out_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            top_p=top_p,
            pad_token_id=tok.eos_token_id,
        )

    return tok.decode(out_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

In [10]:
%%time
# Simple smoke tests
print("\n--- Test 1 ---")
print(ask("Say hello in one short sentence."))

print("\n--- Test 2 (tiny code task) ---")
print(ask("Write a Python function reverse_str(s) that returns the reversed string, then show a quick example."))


--- Test 1 ---
"Hello!"

--- Test 2 (tiny code task) ---
Sure! Here's a Python function that reverses a string:
```
def reverse_str(s):
    return s[::-1]
```
Here's a quick example of how to use the function:
```
>>> reverse_str("hello")
"lohel"
```
The `[::-1]` syntax is used to reverse the string. It works by slicing the string from the end to the beginning, and then returning the resulting slice.

Alternatively, you can use the `reversed()` function to reverse a string:
```
CPU times: user 3.15 s, sys: 186 ms, total: 3.33 s
Wall time: 3.56 s


In [17]:
import torch

def ask_fast(prompt, max_new_tokens=128):
    inst = f"<s>[INST] {prompt.strip()} [/INST]"
    inputs = tok(inst, return_tensors="pt").to(model.device)
    with torch.inference_mode():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            num_beams=1,
            pad_token_id=tok.eos_token_id,
            use_cache=True,
        )
    return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

In [18]:
%%time
# Simple smoke tests
print("\n--- Test 1 ---")
print(ask_fast("Say hello in one short sentence."))

print("\n--- Test 2 (tiny code task) ---")
print(ask_fast("Write a Python function reverse_str(s) that returns the reversed string, then show a quick example."))


--- Test 1 ---
Hello!

--- Test 2 (tiny code task) ---
Sure! Here's a Python function that reverses a string:
```
def reverse_str(s):
    return s[::-1]
```
Here's a quick example of how to use the function:
```
>>> reverse_str("hello")
"lohel"
```
The `[::-1]` syntax is used to reverse the string. It works by slicing the string from the end to the beginning, and then returning the reversed string.

You can also use the `reversed()` function to reverse a string, like this:
CPU times: user 2.92 s, sys: 2.64 ms, total: 2.92 s
Wall time: 2.95 s


Alright, this is working. Let's try to make it process a JSON file.


A very naive way to do it is by "zero-shot"-ing the model (e.g. send everything and hope for the best):

In [19]:
%%time
print(ask("""
Given this entities:
- particle
- geometric_shape
- line
- sound_sample
- rhythm
- sound_envelop
- oscillator
- video
- image
- text
- other

Read this code:

```
var xoff = 0.0
var xinc = 42

function hal() {
    let resx, resy, stepx, stepy, x, y, x1, y1, coords, p1, p2, p3, p4
    resx = 2 * Math.floor(random(2, 5)); resy = 2 * Math.floor(random(2, 5)); coords = []
    stepx = Math.floor(actualwidth / resx); stepy = Math.floor(actualheight / resy)
    x = leftmargin
    for (let i = 0; i < resx; i++) {
        y = topmargin
        for (let j = 0; j < resy; j++) {
            x1 = x + stepx * noise(xoff); xoff += xinc; y1 = y + stepy * noise(xoff)
            coords.push({ x: x1, y: y1 })
            y += stepy}
        x += stepx}
    for (let i = 0; i < resx - 1; i++) {
        for (let j = 0; j < resy - 1; j++) {
            p1 = coords[i * resy + j]; p2 = coords[i * resy + j + resy]; p3 = coords[i * resy + j + resy + 1]; p4 = coords[i * resy + j + 1]
            trianglewlines(p1.x, p1.y, p2.x, p2.y, p4.x, p4.y);
            trianglewlines(p2.x, p2.y, p3.x, p3.y, p4.x, p4.y);
}}}

function trianglewlines(x1, y1, x2, y2, x3, y3) {
    // https://mathopenref.com/coordincenter.html
    len1 = dist(x2, y2, x3, y3); len2 = dist(x1, y1, x3, y3); len3 = dist(x1, y1, x2, y2)
    cx = (len1 * x1 + len2 * x2 + len3 * x3) / (len1 + len2 + len3); cy = (len1 * y1 + len2 * y2 + len3 * y3) / (len1 + len2 + len3)
    for (let i = 0; i < 1; i += 0.03) {
        triangle(lerp(cx, x1, i), lerp(cy, y1, i), lerp(cx, x2, i), lerp(cy, y2, i), lerp(cx, x3, i), lerp(cy, y3, i),
)}}
```

Now select the entities that are present in this code
"""))


The entities present in this code are:

* `particle`
* `geometric_shape`
* `line`
* `sound_sample`
* `rhythm`
* `sound_envelop`
* `oscillator`
* `video`
* `image`
* `text`
* `other`

These entities are used in the code to create a visual representation of a fractal pattern, and to generate sound samples based on the pattern.
CPU times: user 3.25 s, sys: 82.8 ms, total: 3.33 s
Wall time: 3.36 s


It doesn't work - and it's very inneficient. We cannot have 3 seconds per file

What if we make the model answer specific questions?

In [33]:
%%time
print(ask("""
Considering this code:
```
var xoff = 0.0
var xinc = 42

function hal() {
    let resx, resy, stepx, stepy, x, y, x1, y1, coords, p1, p2, p3, p4
    resx = 2 * Math.floor(random(2, 5)); resy = 2 * Math.floor(random(2, 5)); coords = []
    stepx = Math.floor(actualwidth / resx); stepy = Math.floor(actualheight / resy)
    x = leftmargin
    for (let i = 0; i < resx; i++) {
        y = topmargin
        for (let j = 0; j < resy; j++) {
            x1 = x + stepx * noise(xoff); xoff += xinc; y1 = y + stepy * noise(xoff)
            coords.push({ x: x1, y: y1 })
            y += stepy}
        x += stepx}
    for (let i = 0; i < resx - 1; i++) {
        for (let j = 0; j < resy - 1; j++) {
            p1 = coords[i * resy + j]; p2 = coords[i * resy + j + resy]; p3 = coords[i * resy + j + resy + 1]; p4 = coords[i * resy + j + 1]
            trianglewlines(p1.x, p1.y, p2.x, p2.y, p4.x, p4.y);
            trianglewlines(p2.x, p2.y, p3.x, p3.y, p4.x, p4.y);
}}}

function trianglewlines(x1, y1, x2, y2, x3, y3) {
    // https://mathopenref.com/coordincenter.html
    len1 = dist(x2, y2, x3, y3); len2 = dist(x1, y1, x3, y3); len3 = dist(x1, y1, x2, y2)
    cx = (len1 * x1 + len2 * x2 + len3 * x3) / (len1 + len2 + len3); cy = (len1 * y1 + len2 * y2 + len3 * y3) / (len1 + len2 + len3)
    for (let i = 0; i < 1; i += 0.03) {
        triangle(lerp(cx, x1, i), lerp(cy, y1, i), lerp(cx, x2, i), lerp(cy, y2, i), lerp(cx, x3, i), lerp(cy, y3, i),
)}}
```
Create a Python list with boolean values with the answers for the following questions:

1) Does it contain lines? [True/False]
2) Does it contain particles? [True/False]

Answers:
[ , ]
"""))


Based on the provided JavaScript code, the answers to the questions are:

1. Does it contain lines? - True
2. Does it contain particles? - False
CPU times: user 1.37 s, sys: 71.3 ms, total: 1.44 s
Wall time: 1.46 s


It's faster, not as fast as we need, but we wont be able to parse this.

---

In [52]:
%%timeit
print(ask_fast("""
Considering this code:
```
var xoff = 0.0
var xinc = 42

function hal() {
    let resx, resy, stepx, stepy, x, y, x1, y1, coords, p1, p2, p3, p4
    resx = 2 * Math.floor(random(2, 5)); resy = 2 * Math.floor(random(2, 5)); coords = []
    stepx = Math.floor(actualwidth / resx); stepy = Math.floor(actualheight / resy)
    x = leftmargin
    for (let i = 0; i < resx; i++) {
        y = topmargin
        for (let j = 0; j < resy; j++) {
            x1 = x + stepx * noise(xoff); xoff += xinc; y1 = y + stepy * noise(xoff)
            coords.push({ x: x1, y: y1 })
            y += stepy}
        x += stepx}
    for (let i = 0; i < resx - 1; i++) {
        for (let j = 0; j < resy - 1; j++) {
            p1 = coords[i * resy + j]; p2 = coords[i * resy + j + resy]; p3 = coords[i * resy + j + resy + 1]; p4 = coords[i * resy + j + 1]
            trianglewlines(p1.x, p1.y, p2.x, p2.y, p4.x, p4.y);
            trianglewlines(p2.x, p2.y, p3.x, p3.y, p4.x, p4.y);
}}}

function trianglewlines(x1, y1, x2, y2, x3, y3) {
    // https://mathopenref.com/coordincenter.html
    len1 = dist(x2, y2, x3, y3); len2 = dist(x1, y1, x3, y3); len3 = dist(x1, y1, x2, y2)
    cx = (len1 * x1 + len2 * x2 + len3 * x3) / (len1 + len2 + len3); cy = (len1 * y1 + len2 * y2 + len3 * y3) / (len1 + len2 + len3)
    for (let i = 0; i < 1; i += 0.03) {
        triangle(lerp(cx, x1, i), lerp(cy, y1, i), lerp(cx, x2, i), lerp(cy, y2, i), lerp(cx, x3, i), lerp(cy, y3, i),
)}}
```
Create a Python list with boolean values with the answers for the following questions:

1) Does it contain lines? [True/False]
2) Does it contain particles? [True/False]

Now write the python list object with the answers and nothing else:
"""))


[True, False]
[True, False]
[True, False]
[True, False]
[True, False]
[True, False]
[True, False]
[True, False]
680 ms ± 222 μs per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [51]:
%%timeit
print(ask("""
Considering this code:
```
var xoff = 0.0
var xinc = 42

function hal() {
    let resx, resy, stepx, stepy, x, y, x1, y1, coords, p1, p2, p3, p4
    resx = 2 * Math.floor(random(2, 5)); resy = 2 * Math.floor(random(2, 5)); coords = []
    stepx = Math.floor(actualwidth / resx); stepy = Math.floor(actualheight / resy)
    x = leftmargin
    for (let i = 0; i < resx; i++) {
        y = topmargin
        for (let j = 0; j < resy; j++) {
            x1 = x + stepx * noise(xoff); xoff += xinc; y1 = y + stepy * noise(xoff)
            coords.push({ x: x1, y: y1 })
            y += stepy}
        x += stepx}
    for (let i = 0; i < resx - 1; i++) {
        for (let j = 0; j < resy - 1; j++) {
            p1 = coords[i * resy + j]; p2 = coords[i * resy + j + resy]; p3 = coords[i * resy + j + resy + 1]; p4 = coords[i * resy + j + 1]
            trianglewlines(p1.x, p1.y, p2.x, p2.y, p4.x, p4.y);
            trianglewlines(p2.x, p2.y, p3.x, p3.y, p4.x, p4.y);
}}}

function trianglewlines(x1, y1, x2, y2, x3, y3) {
    // https://mathopenref.com/coordincenter.html
    len1 = dist(x2, y2, x3, y3); len2 = dist(x1, y1, x3, y3); len3 = dist(x1, y1, x2, y2)
    cx = (len1 * x1 + len2 * x2 + len3 * x3) / (len1 + len2 + len3); cy = (len1 * y1 + len2 * y2 + len3 * y3) / (len1 + len2 + len3)
    for (let i = 0; i < 1; i += 0.03) {
        triangle(lerp(cx, x1, i), lerp(cy, y1, i), lerp(cx, x2, i), lerp(cy, y2, i), lerp(cx, x3, i), lerp(cy, y3, i),
)}}
```
Create a JSON list with boolean values with the answers for the following questions:

1) Does it contain lines? [True/False]
2) Does it contain particles? [True/False]

Now write the enumerated json list object with answers and nothing else:
"""))

{
"1": true,
"2": false
}
{
"1": true,
"2": false
}
Here is the JSON list with boolean values for the questions:

[
  {
    "question": "Does it contain lines?",
    "answer": true
  },
  {
    "question": "Does it contain particles?",
    "answer": false
  }
]
{
"1": true,
"2": false
}
{
"1": true,
"2": false
}
{
"1": true,
"2": false
}
{
"1": true,
"2": false
}
Here is the JSON list with boolean values for the questions:
```
[
  {
    "question": "Does it contain lines?",
    "answer": true
  },
  {
    "question": "Does it contain particles?",
    "answer": false
  }
]
```
1.3 s ± 616 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [50]:
%%timeit
print(ask_fast("""
Considering this code:
```
var xoff = 0.0
var xinc = 42

function hal() {
    let resx, resy, stepx, stepy, x, y, x1, y1, coords, p1, p2, p3, p4
    resx = 2 * Math.floor(random(2, 5)); resy = 2 * Math.floor(random(2, 5)); coords = []
    stepx = Math.floor(actualwidth / resx); stepy = Math.floor(actualheight / resy)
    x = leftmargin
    for (let i = 0; i < resx; i++) {
        y = topmargin
        for (let j = 0; j < resy; j++) {
            x1 = x + stepx * noise(xoff); xoff += xinc; y1 = y + stepy * noise(xoff)
            coords.push({ x: x1, y: y1 })
            y += stepy}
        x += stepx}
    for (let i = 0; i < resx - 1; i++) {
        for (let j = 0; j < resy - 1; j++) {
            p1 = coords[i * resy + j]; p2 = coords[i * resy + j + resy]; p3 = coords[i * resy + j + resy + 1]; p4 = coords[i * resy + j + 1]
            trianglewlines(p1.x, p1.y, p2.x, p2.y, p4.x, p4.y);
            trianglewlines(p2.x, p2.y, p3.x, p3.y, p4.x, p4.y);
}}}

function trianglewlines(x1, y1, x2, y2, x3, y3) {
    // https://mathopenref.com/coordincenter.html
    len1 = dist(x2, y2, x3, y3); len2 = dist(x1, y1, x3, y3); len3 = dist(x1, y1, x2, y2)
    cx = (len1 * x1 + len2 * x2 + len3 * x3) / (len1 + len2 + len3); cy = (len1 * y1 + len2 * y2 + len3 * y3) / (len1 + len2 + len3)
    for (let i = 0; i < 1; i += 0.03) {
        triangle(lerp(cx, x1, i), lerp(cy, y1, i), lerp(cx, x2, i), lerp(cy, y2, i), lerp(cx, x3, i), lerp(cy, y3, i),
)}}
```
Create a JSON list with boolean values with the answers for the following questions:

1) Does it contain lines? [True/False]
2) Does it contain particles? [True/False]

Now write the enumerated json list object with answers and nothing else:
"""))

{
"1": true,
"2": false
}
{
"1": true,
"2": false
}
{
"1": true,
"2": false
}
{
"1": true,
"2": false
}
{
"1": true,
"2": false
}
{
"1": true,
"2": false
}
{
"1": true,
"2": false
}
{
"1": true,
"2": false
}
914 ms ± 13.3 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


---

Trying to make the inference faster with quantization

https://arxiv.org/pdf/2305.14314 *QLORA introduces a number of innovations to save memory
without sacrificing performance: (a) 4-bit NormalFloat (NF4), a new data type that
is information theoretically optimal for normally distributed weights (b) Double
Quantization to reduce the average memory footprint by quantizing the quantization
constants, and (c) Paged Optimizers to manage memory spikes*

In [13]:
from transformers import BitsAndBytesConfig

q_config = BitsAndBytesConfig(
   load_in_4bit=True,
   bnb_4bit_quant_type="nf4",
   bnb_4bit_use_double_quant=True,
   bnb_4bit_compute_dtype=torch.bfloat16
) 

model = AutoModelForCausalLM.from_pretrained(
        MODEL_PATH,
        local_files_only=True,
        device_map="auto",
        low_cpu_mem_usage=True,
        quantization_config=q_config, # new param to control quantization
    )

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [8]:
from transformers import BitsAndBytesConfig

q_config = BitsAndBytesConfig(
   load_in_4bit=True,
   bnb_4bit_quant_type="nf4",
   bnb_4bit_use_double_quant=True,
   bnb_4bit_compute_dtype=torch.bfloat16
) 

In [15]:
model = AutoModelForCausalLM.from_pretrained(
        MODEL_PATH,
        local_files_only=True,
        device_map="auto",
        low_cpu_mem_usage=True,
        quantization_config=q_config, # new param to control quantization
    )

def ask_q(prompt, max_new_tokens=128):
    inst = f"<s>[INST] {prompt.strip()} [/INST]"
    inputs = tok(inst, return_tensors="pt").to(model.device)
    with torch.inference_mode():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            num_beams=1,
            pad_token_id=tok.eos_token_id,
            
            use_cache=True,
        )
    return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

In [16]:
%%timeit
print(ask_q("""
Considering this code:
```
var xoff = 0.0
var xinc = 42

function hal() {
    let resx, resy, stepx, stepy, x, y, x1, y1, coords, p1, p2, p3, p4
    resx = 2 * Math.floor(random(2, 5)); resy = 2 * Math.floor(random(2, 5)); coords = []
    stepx = Math.floor(actualwidth / resx); stepy = Math.floor(actualheight / resy)
    x = leftmargin
    for (let i = 0; i < resx; i++) {
        y = topmargin
        for (let j = 0; j < resy; j++) {
            x1 = x + stepx * noise(xoff); xoff += xinc; y1 = y + stepy * noise(xoff)
            coords.push({ x: x1, y: y1 })
            y += stepy}
        x += stepx}
    for (let i = 0; i < resx - 1; i++) {
        for (let j = 0; j < resy - 1; j++) {
            p1 = coords[i * resy + j]; p2 = coords[i * resy + j + resy]; p3 = coords[i * resy + j + resy + 1]; p4 = coords[i * resy + j + 1]
            trianglewlines(p1.x, p1.y, p2.x, p2.y, p4.x, p4.y);
            trianglewlines(p2.x, p2.y, p3.x, p3.y, p4.x, p4.y);
}}}

function trianglewlines(x1, y1, x2, y2, x3, y3) {
    // https://mathopenref.com/coordincenter.html
    len1 = dist(x2, y2, x3, y3); len2 = dist(x1, y1, x3, y3); len3 = dist(x1, y1, x2, y2)
    cx = (len1 * x1 + len2 * x2 + len3 * x3) / (len1 + len2 + len3); cy = (len1 * y1 + len2 * y2 + len3 * y3) / (len1 + len2 + len3)
    for (let i = 0; i < 1; i += 0.03) {
        triangle(lerp(cx, x1, i), lerp(cy, y1, i), lerp(cx, x2, i), lerp(cy, y2, i), lerp(cx, x3, i), lerp(cy, y3, i),
)}}
```
Create a JSON list with boolean values with the answers for the following questions:

1) Does it contain lines? [True/False]
2) Does it contain particles? [True/False]

Now write the enumerated json list object with answers and nothing else:
"""))

{
"1": true,
"2": false
}
{
"1": true,
"2": false
}
{
"1": true,
"2": false
}
{
"1": true,
"2": false
}
{
"1": true,
"2": false
}
{
"1": true,
"2": false
}
{
"1": true,
"2": false
}
{
"1": true,
"2": false
}
556 ms ± 9.48 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [23]:
print(f"{round((914 - 556)*100 / 914, 2)}% faster inference time")

39.17% faster inference time


---

Can we make it even faster by compiling the model and using flash attention?


https://arxiv.org/pdf/2205.14135 *We propose FlashAttention, an IO-aware exact attention algorithm that uses tiling to reduce the number of memory reads/writes between GPU high bandwidth memory (HBM) and GPU on-chip SRAM*

In [3]:
from transformers import BitsAndBytesConfig

q_config = BitsAndBytesConfig(
   load_in_4bit=True,
   bnb_4bit_quant_type="nf4",
   bnb_4bit_use_double_quant=True,
   bnb_4bit_compute_dtype=torch.bfloat16
) 

model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    local_files_only=True,
    device_map="auto",
    low_cpu_mem_usage=True,
    quantization_config=q_config,
    attn_implementation="sdpa", # flash attention
    dtype=torch.bfloat16, # and changing dtype to lower precision
)

def ask_q(prompt, max_new_tokens=128):
    inst = f"<s>[INST] {prompt.strip()} [/INST]"
    inputs = tok(inst, return_tensors="pt").to(model.device)
    with torch.inference_mode():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            num_beams=1,
            pad_token_id=tok.eos_token_id,
            use_cache=True,
        )
    return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [4]:
%%timeit
print(ask_q("""
Considering this code:
```
var xoff = 0.0
var xinc = 42

function hal() {
    let resx, resy, stepx, stepy, x, y, x1, y1, coords, p1, p2, p3, p4
    resx = 2 * Math.floor(random(2, 5)); resy = 2 * Math.floor(random(2, 5)); coords = []
    stepx = Math.floor(actualwidth / resx); stepy = Math.floor(actualheight / resy)
    x = leftmargin
    for (let i = 0; i < resx; i++) {
        y = topmargin
        for (let j = 0; j < resy; j++) {
            x1 = x + stepx * noise(xoff); xoff += xinc; y1 = y + stepy * noise(xoff)
            coords.push({ x: x1, y: y1 })
            y += stepy}
        x += stepx}
    for (let i = 0; i < resx - 1; i++) {
        for (let j = 0; j < resy - 1; j++) {
            p1 = coords[i * resy + j]; p2 = coords[i * resy + j + resy]; p3 = coords[i * resy + j + resy + 1]; p4 = coords[i * resy + j + 1]
            trianglewlines(p1.x, p1.y, p2.x, p2.y, p4.x, p4.y);
            trianglewlines(p2.x, p2.y, p3.x, p3.y, p4.x, p4.y);
}}}

function trianglewlines(x1, y1, x2, y2, x3, y3) {
    // https://mathopenref.com/coordincenter.html
    len1 = dist(x2, y2, x3, y3); len2 = dist(x1, y1, x3, y3); len3 = dist(x1, y1, x2, y2)
    cx = (len1 * x1 + len2 * x2 + len3 * x3) / (len1 + len2 + len3); cy = (len1 * y1 + len2 * y2 + len3 * y3) / (len1 + len2 + len3)
    for (let i = 0; i < 1; i += 0.03) {
        triangle(lerp(cx, x1, i), lerp(cy, y1, i), lerp(cx, x2, i), lerp(cy, y2, i), lerp(cx, x3, i), lerp(cy, y3, i),
)}}
```
Create a JSON list with boolean values with the answers for the following questions:

1) Does it contain lines? [True/False]
2) Does it contain particles? [True/False]

Now write the enumerated json list object with answers and nothing else:
"""))

{
"1": true,
"2": false
}
{
"1": true,
"2": false
}
{
"1": true,
"2": false
}
{
"1": true,
"2": false
}
{
"1": true,
"2": false
}
{
"1": true,
"2": false
}
{
"1": true,
"2": false
}
{
"1": true,
"2": false
}
455 ms ± 3.04 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [5]:
print(f"{round((914 - 455)*100 / 914, 2)}% faster inference time")

50.22% faster inference time


---

In [16]:
%%timeit
print(ask_q("""
Considering this code:
```
var xoff = 0.0
var xinc = 42

function hal() {
    let resx, resy, stepx, stepy, x, y, x1, y1, coords, p1, p2, p3, p4
    resx = 2 * Math.floor(random(2, 5)); resy = 2 * Math.floor(random(2, 5)); coords = []
    stepx = Math.floor(actualwidth / resx); stepy = Math.floor(actualheight / resy)
    x = leftmargin
    for (let i = 0; i < resx; i++) {
        y = topmargin
        for (let j = 0; j < resy; j++) {
            x1 = x + stepx * noise(xoff); xoff += xinc; y1 = y + stepy * noise(xoff)
            coords.push({ x: x1, y: y1 })
            y += stepy}
        x += stepx}
    for (let i = 0; i < resx - 1; i++) {
        for (let j = 0; j < resy - 1; j++) {
            p1 = coords[i * resy + j]; p2 = coords[i * resy + j + resy]; p3 = coords[i * resy + j + resy + 1]; p4 = coords[i * resy + j + 1]
            trianglewlines(p1.x, p1.y, p2.x, p2.y, p4.x, p4.y);
            trianglewlines(p2.x, p2.y, p3.x, p3.y, p4.x, p4.y);
}}}

function trianglewlines(x1, y1, x2, y2, x3, y3) {
    // https://mathopenref.com/coordincenter.html
    len1 = dist(x2, y2, x3, y3); len2 = dist(x1, y1, x3, y3); len3 = dist(x1, y1, x2, y2)
    cx = (len1 * x1 + len2 * x2 + len3 * x3) / (len1 + len2 + len3); cy = (len1 * y1 + len2 * y2 + len3 * y3) / (len1 + len2 + len3)
    for (let i = 0; i < 1; i += 0.03) {
        triangle(lerp(cx, x1, i), lerp(cy, y1, i), lerp(cx, x2, i), lerp(cy, y2, i), lerp(cx, x3, i), lerp(cy, y3, i),
)}}
```
Create a JSON list with boolean values with the answers for the following questions:

1) Does it produce audio file? [True/False]
2) Does it produce image file? [True/False]
3) Does it produce synthesized sound? [True/False]
4) Does it produce synthesized image? [True/False]
5) Does it contain randomness? [True/False]
6) Does it contain interactions? [True/False]

Now write the enumerated json list object with answers and nothing else:
"""))

{
"1": false,
"2": true,
"3": false,
"4": false,
"5": true,
"6": false
}
{
"1": false,
"2": true,
"3": false,
"4": false,
"5": true,
"6": false
}
{
"1": false,
"2": true,
"3": false,
"4": false,
"5": true,
"6": false
}
{
"1": false,
"2": true,
"3": false,
"4": false,
"5": true,
"6": false
}
{
"1": false,
"2": true,
"3": false,
"4": false,
"5": true,
"6": false
}
{
"1": false,
"2": true,
"3": false,
"4": false,
"5": true,
"6": false
}
{
"1": false,
"2": true,
"3": false,
"4": false,
"5": true,
"6": false
}
{
"1": false,
"2": true,
"3": false,
"4": false,
"5": true,
"6": false
}
1.08 s ± 19.6 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


Let's keep increasing the number of questions:

In [14]:
%%timeit
print(ask_q("""
Considering this code:
```
var xoff = 0.0
var xinc = 42

function hal() {
    let resx, resy, stepx, stepy, x, y, x1, y1, coords, p1, p2, p3, p4
    resx = 2 * Math.floor(random(2, 5)); resy = 2 * Math.floor(random(2, 5)); coords = []
    stepx = Math.floor(actualwidth / resx); stepy = Math.floor(actualheight / resy)
    x = leftmargin
    for (let i = 0; i < resx; i++) {
        y = topmargin
        for (let j = 0; j < resy; j++) {
            x1 = x + stepx * noise(xoff); xoff += xinc; y1 = y + stepy * noise(xoff)
            coords.push({ x: x1, y: y1 })
            y += stepy}
        x += stepx}
    for (let i = 0; i < resx - 1; i++) {
        for (let j = 0; j < resy - 1; j++) {
            p1 = coords[i * resy + j]; p2 = coords[i * resy + j + resy]; p3 = coords[i * resy + j + resy + 1]; p4 = coords[i * resy + j + 1]
            trianglewlines(p1.x, p1.y, p2.x, p2.y, p4.x, p4.y);
            trianglewlines(p2.x, p2.y, p3.x, p3.y, p4.x, p4.y);
}}}

function trianglewlines(x1, y1, x2, y2, x3, y3) {
    // https://mathopenref.com/coordincenter.html
    len1 = dist(x2, y2, x3, y3); len2 = dist(x1, y1, x3, y3); len3 = dist(x1, y1, x2, y2)
    cx = (len1 * x1 + len2 * x2 + len3 * x3) / (len1 + len2 + len3); cy = (len1 * y1 + len2 * y2 + len3 * y3) / (len1 + len2 + len3)
    for (let i = 0; i < 1; i += 0.03) {
        triangle(lerp(cx, x1, i), lerp(cy, y1, i), lerp(cx, x2, i), lerp(cy, y2, i), lerp(cx, x3, i), lerp(cy, y3, i),
)}}
```
Create a JSON list with boolean values with the answers for the following questions:

1) Does it produce audio file? [True/False]
2) Does it produce image file? [True/False]
3) Does it produce synthesized sound? [True/False]
4) Does it produce synthesized image? [True/False]
5) Does it contain randomness? [True/False]
6) Does it contain interactions? [True/False]
7) Does it have human interaction (for example through mouse, midi controller, microphone, keyboard, camera, motion sensor, lidar, etc.)? [True/False]
8) Does it have computer interaction (for example through data file, data stream, remote data, web API, etc.)? [True/False]
9) Does it uses visual sensory outcomes? [True/False]
10) Does it uses visual sensory outcomes? [True/False]
11) Does it uses auditory sensory outcomes? [True/False]
12) Does it uses physical sensory outcomes? [True/False]
13) Does it uses static or time-based [one of these values is mandatory for this characteristic] sensory outcomes? [True/False]

Now write the enumerated-index json list object with answers and nothing else:

"""))

{
    "audio": false,
    "image": false,
    "synthesized sound": false,
    "synthesized image": false,
    "randomness": true,
    "interactions": false,
    "human interaction": false,
    "computer interaction": false,
    "visual sensory outcomes": true,
    "auditory sensory outcomes": false,
    "physical sensory outcomes": false,
    "static or time-based sensory outcomes": false
  }
{
    "audio": false,
    "image": false,
    "synthesized sound": false,
    "synthesized image": false,
    "randomness": true,
    "interactions": false,
    "human interaction": false,
    "computer interaction": false,
    "visual sensory outcomes": true,
    "auditory sensory outcomes": false,
    "physical sensory outcomes": false,
    "static or time-based sensory outcomes": false
  }
{
    "audio": false,
    "image": false,
    "synthesized sound": false,
    "synthesized image": false,
    "randomness": true,
    "interactions": false,
    "human interaction": false,
    "computer inter

The output changed and now it's taking more time to generate a response. We should keep it just a few questions at a time. Or even one at a time...

In [17]:
%%timeit
print(ask_q("""
Considering this code:
```
var xoff = 0.0
var xinc = 42

function hal() {
    let resx, resy, stepx, stepy, x, y, x1, y1, coords, p1, p2, p3, p4
    resx = 2 * Math.floor(random(2, 5)); resy = 2 * Math.floor(random(2, 5)); coords = []
    stepx = Math.floor(actualwidth / resx); stepy = Math.floor(actualheight / resy)
    x = leftmargin
    for (let i = 0; i < resx; i++) {
        y = topmargin
        for (let j = 0; j < resy; j++) {
            x1 = x + stepx * noise(xoff); xoff += xinc; y1 = y + stepy * noise(xoff)
            coords.push({ x: x1, y: y1 })
            y += stepy}
        x += stepx}
    for (let i = 0; i < resx - 1; i++) {
        for (let j = 0; j < resy - 1; j++) {
            p1 = coords[i * resy + j]; p2 = coords[i * resy + j + resy]; p3 = coords[i * resy + j + resy + 1]; p4 = coords[i * resy + j + 1]
            trianglewlines(p1.x, p1.y, p2.x, p2.y, p4.x, p4.y);
            trianglewlines(p2.x, p2.y, p3.x, p3.y, p4.x, p4.y);
}}}

function trianglewlines(x1, y1, x2, y2, x3, y3) {
    // https://mathopenref.com/coordincenter.html
    len1 = dist(x2, y2, x3, y3); len2 = dist(x1, y1, x3, y3); len3 = dist(x1, y1, x2, y2)
    cx = (len1 * x1 + len2 * x2 + len3 * x3) / (len1 + len2 + len3); cy = (len1 * y1 + len2 * y2 + len3 * y3) / (len1 + len2 + len3)
    for (let i = 0; i < 1; i += 0.03) {
        triangle(lerp(cx, x1, i), lerp(cy, y1, i), lerp(cx, x2, i), lerp(cy, y2, i), lerp(cx, x3, i), lerp(cy, y3, i),
)}}
```
Create a JSON list with boolean values with the answers for the following questions:

1) Does it produce audio file? [True/False]

Now write the enumerated-index json list object with answers and nothing else:
"""))

{
"1": false
}
{
"1": false
}
{
"1": false
}
{
"1": false
}
{
"1": false
}
{
"1": false
}
{
"1": false
}
{
"1": false
}
303 ms ± 2.21 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


Trying another category

In [22]:
%%timeit
print(ask_q("""
Considering this code:
```
var xoff = 0.0
var xinc = 42

function hal() {
    let resx, resy, stepx, stepy, x, y, x1, y1, coords, p1, p2, p3, p4
    resx = 2 * Math.floor(random(2, 5)); resy = 2 * Math.floor(random(2, 5)); coords = []
    stepx = Math.floor(actualwidth / resx); stepy = Math.floor(actualheight / resy)
    x = leftmargin
    for (let i = 0; i < resx; i++) {
        y = topmargin
        for (let j = 0; j < resy; j++) {
            x1 = x + stepx * noise(xoff); xoff += xinc; y1 = y + stepy * noise(xoff)
            coords.push({ x: x1, y: y1 })
            y += stepy}
        x += stepx}
    for (let i = 0; i < resx - 1; i++) {
        for (let j = 0; j < resy - 1; j++) {
            p1 = coords[i * resy + j]; p2 = coords[i * resy + j + resy]; p3 = coords[i * resy + j + resy + 1]; p4 = coords[i * resy + j + 1]
            trianglewlines(p1.x, p1.y, p2.x, p2.y, p4.x, p4.y);
            trianglewlines(p2.x, p2.y, p3.x, p3.y, p4.x, p4.y);
}}}

function trianglewlines(x1, y1, x2, y2, x3, y3) {
    // https://mathopenref.com/coordincenter.html
    len1 = dist(x2, y2, x3, y3); len2 = dist(x1, y1, x3, y3); len3 = dist(x1, y1, x2, y2)
    cx = (len1 * x1 + len2 * x2 + len3 * x3) / (len1 + len2 + len3); cy = (len1 * y1 + len2 * y2 + len3 * y3) / (len1 + len2 + len3)
    for (let i = 0; i < 1; i += 0.03) {
        triangle(lerp(cx, x1, i), lerp(cy, y1, i), lerp(cx, x2, i), lerp(cy, y2, i), lerp(cx, x3, i), lerp(cy, y3, i),
)}}
```
Create a JSON list with boolean values with the answers for the following questions:

1) Does it have human interaction (for example through mouse, midi controller, microphone, keyboard, camera, motion sensor, lidar)? [True/False]
2) Does it have computer interaction (for example through data file, data stream, remote data, web API)? [True/False]

Now write the enumerated-index json list object with answers and nothing else with the following template:
"""))

[
{
"question": "Does it have human interaction (for example through mouse, midi controller, microphone, keyboard, camera, motion sensor, lidar)?",
"answer": true
},
{
"question": "Does it have computer interaction (for example through data file, data stream, remote data, web API)?",
"answer": false
}
]
[
{
"question": "Does it have human interaction (for example through mouse, midi controller, microphone, keyboard, camera, motion sensor, lidar)?",
"answer": true
},
{
"question": "Does it have computer interaction (for example through data file, data stream, remote data, web API)?",
"answer": false
}
]
[
{
"question": "Does it have human interaction (for example through mouse, midi controller, microphone, keyboard, camera, motion sensor, lidar)?",
"answer": true
},
{
"question": "Does it have computer interaction (for example through data file, data stream, remote data, web API)?",
"answer": false
}
]
[
{
"question": "Does it have human interaction (for example through mouse, midi cont

Oh, a more verbose question triggered a verbose json. Let's pass a template json

In [21]:
%%timeit
print(ask_q("""
Considering this code:
```
var xoff = 0.0
var xinc = 42

function hal() {
    let resx, resy, stepx, stepy, x, y, x1, y1, coords, p1, p2, p3, p4
    resx = 2 * Math.floor(random(2, 5)); resy = 2 * Math.floor(random(2, 5)); coords = []
    stepx = Math.floor(actualwidth / resx); stepy = Math.floor(actualheight / resy)
    x = leftmargin
    for (let i = 0; i < resx; i++) {
        y = topmargin
        for (let j = 0; j < resy; j++) {
            x1 = x + stepx * noise(xoff); xoff += xinc; y1 = y + stepy * noise(xoff)
            coords.push({ x: x1, y: y1 })
            y += stepy}
        x += stepx}
    for (let i = 0; i < resx - 1; i++) {
        for (let j = 0; j < resy - 1; j++) {
            p1 = coords[i * resy + j]; p2 = coords[i * resy + j + resy]; p3 = coords[i * resy + j + resy + 1]; p4 = coords[i * resy + j + 1]
            trianglewlines(p1.x, p1.y, p2.x, p2.y, p4.x, p4.y);
            trianglewlines(p2.x, p2.y, p3.x, p3.y, p4.x, p4.y);
}}}

function trianglewlines(x1, y1, x2, y2, x3, y3) {
    // https://mathopenref.com/coordincenter.html
    len1 = dist(x2, y2, x3, y3); len2 = dist(x1, y1, x3, y3); len3 = dist(x1, y1, x2, y2)
    cx = (len1 * x1 + len2 * x2 + len3 * x3) / (len1 + len2 + len3); cy = (len1 * y1 + len2 * y2 + len3 * y3) / (len1 + len2 + len3)
    for (let i = 0; i < 1; i += 0.03) {
        triangle(lerp(cx, x1, i), lerp(cy, y1, i), lerp(cx, x2, i), lerp(cy, y2, i), lerp(cx, x3, i), lerp(cy, y3, i),
)}}
```
Create a JSON list with boolean values with the answers for the following questions:

1) Does it have human interaction (for example through mouse, midi controller, microphone, keyboard, camera, motion sensor, lidar)? [True/False]
2) Does it have computer interaction (for example through data file, data stream, remote data, web API)? [True/False]

Now write the enumerated-index json list object with answers and nothing else with the following template:

{
    "1": <True/False>,
    "2": <True/False>
}
"""))

{
    "1": False,
    "2": True
}
{
    "1": False,
    "2": True
}
{
    "1": False,
    "2": True
}
{
    "1": False,
    "2": True
}
{
    "1": False,
    "2": True
}
{
    "1": False,
    "2": True
}
{
    "1": False,
    "2": True
}
{
    "1": False,
    "2": True
}
509 ms ± 1.66 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [47]:
%%timeit
print(ask_q("""
Considering this code:
```
var xoff = 0.0
var xinc = 42

function hal() {
    let resx, resy, stepx, stepy, x, y, x1, y1, coords, p1, p2, p3, p4
    resx = 2 * Math.floor(random(2, 5)); resy = 2 * Math.floor(random(2, 5)); coords = []
    stepx = Math.floor(actualwidth / resx); stepy = Math.floor(actualheight / resy)
    x = leftmargin
    for (let i = 0; i < resx; i++) {
        y = topmargin
        for (let j = 0; j < resy; j++) {
            x1 = x + stepx * noise(xoff); xoff += xinc; y1 = y + stepy * noise(xoff)
            coords.push({ x: x1, y: y1 })
            y += stepy}
        x += stepx}
    for (let i = 0; i < resx - 1; i++) {
        for (let j = 0; j < resy - 1; j++) {
            p1 = coords[i * resy + j]; p2 = coords[i * resy + j + resy]; p3 = coords[i * resy + j + resy + 1]; p4 = coords[i * resy + j + 1]
            trianglewlines(p1.x, p1.y, p2.x, p2.y, p4.x, p4.y);
            trianglewlines(p2.x, p2.y, p3.x, p3.y, p4.x, p4.y);
}}}

function trianglewlines(x1, y1, x2, y2, x3, y3) {
    // https://mathopenref.com/coordincenter.html
    len1 = dist(x2, y2, x3, y3); len2 = dist(x1, y1, x3, y3); len3 = dist(x1, y1, x2, y2)
    cx = (len1 * x1 + len2 * x2 + len3 * x3) / (len1 + len2 + len3); cy = (len1 * y1 + len2 * y2 + len3 * y3) / (len1 + len2 + len3)
    for (let i = 0; i < 1; i += 0.03) {
        triangle(lerp(cx, x1, i), lerp(cy, y1, i), lerp(cx, x2, i), lerp(cy, y2, i), lerp(cx, x3, i), lerp(cy, y3, i),
)}}
```
Create a JSON list with boolean values with the answers for the following questions:

1) Does it utilise external audio file? [True/False]
2) Does it utilise external image file? [True/False]
3) Does it synthesizes sound? [True/False]
4) Does it synthesizes images? [True/False]
5) Does it contain randomness? [True/False]
6) Does it contain interactions? [True/False]

Now write the enumerated-index json list object with answers and nothing else with the following template:

{
    "1": <True/False>,
    "2": <True/False>,
    "3": <True/False>,
    "4": <True/False>,
    "5": <True/False>,
    "6": <True/False>
}
"""))

{
    "1": False,
    "2": False,
    "3": False,
    "4": False,
    "5": True,
    "6": False
}
{
    "1": False,
    "2": False,
    "3": False,
    "4": False,
    "5": True,
    "6": False
}
{
    "1": False,
    "2": False,
    "3": False,
    "4": False,
    "5": True,
    "6": False
}
{
    "1": False,
    "2": False,
    "3": False,
    "4": False,
    "5": True,
    "6": False
}
{
    "1": False,
    "2": False,
    "3": False,
    "4": False,
    "5": True,
    "6": False
}
{
    "1": False,
    "2": False,
    "3": False,
    "4": False,
    "5": True,
    "6": False
}
{
    "1": False,
    "2": False,
    "3": False,
    "4": False,
    "5": True,
    "6": False
}
{
    "1": False,
    "2": False,
    "3": False,
    "4": False,
    "5": True,
    "6": False
}
1.23 s ± 3.58 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [46]:
%%timeit
print(ask_q("""
Considering this code:
```
var xoff = 0.0
var xinc = 42

function hal() {
    let resx, resy, stepx, stepy, x, y, x1, y1, coords, p1, p2, p3, p4
    resx = 2 * Math.floor(random(2, 5)); resy = 2 * Math.floor(random(2, 5)); coords = []
    stepx = Math.floor(actualwidth / resx); stepy = Math.floor(actualheight / resy)
    x = leftmargin
    for (let i = 0; i < resx; i++) {
        y = topmargin
        for (let j = 0; j < resy; j++) {
            x1 = x + stepx * noise(xoff); xoff += xinc; y1 = y + stepy * noise(xoff)
            coords.push({ x: x1, y: y1 })
            y += stepy}
        x += stepx}
    for (let i = 0; i < resx - 1; i++) {
        for (let j = 0; j < resy - 1; j++) {
            p1 = coords[i * resy + j]; p2 = coords[i * resy + j + resy]; p3 = coords[i * resy + j + resy + 1]; p4 = coords[i * resy + j + 1]
            trianglewlines(p1.x, p1.y, p2.x, p2.y, p4.x, p4.y);
            trianglewlines(p2.x, p2.y, p3.x, p3.y, p4.x, p4.y);
}}}

function trianglewlines(x1, y1, x2, y2, x3, y3) {
    // https://mathopenref.com/coordincenter.html
    len1 = dist(x2, y2, x3, y3); len2 = dist(x1, y1, x3, y3); len3 = dist(x1, y1, x2, y2)
    cx = (len1 * x1 + len2 * x2 + len3 * x3) / (len1 + len2 + len3); cy = (len1 * y1 + len2 * y2 + len3 * y3) / (len1 + len2 + len3)
    for (let i = 0; i < 1; i += 0.03) {
        triangle(lerp(cx, x1, i), lerp(cy, y1, i), lerp(cx, x2, i), lerp(cy, y2, i), lerp(cx, x3, i), lerp(cy, y3, i),
)}}
```
Create a JSON list with boolean values with the answers for the following questions:

1) Does it uses visual sensory outcomes? [True/False]
2) Does it uses auditory sensory outcomes? [True/False]
3) Does it uses physical sensory outcomes? [True/False]
4) Does it uses static or time-based [one of these values is mandatory for this characteristic] sensory outcomes? [True/False]

Now write the enumerated-index json list object with answers and nothing else with the following template:

{
    "1": <True/False>,
    "2": <True/False>,
    "3": <True/False>,
    "4": <True/False>
}
"""))

{
    "1": False,
    "2": False,
    "3": True,
    "4": True
}

Explanation:

1. No, the code does not use visual sensory outcomes.
2. No, the code does not use auditory sensory outcomes.
3. Yes, the code uses physical sensory outcomes, specifically the `noise()` function to generate random values for the x-coordinate of the points.
4. Yes, the code uses time-based sensory outcomes, specifically the `xoff` variable
{
    "1": False,
    "2": False,
    "3": True,
    "4": True
}

Explanation:

1. No, the code does not use visual sensory outcomes.
2. No, the code does not use auditory sensory outcomes.
3. Yes, the code uses physical sensory outcomes, specifically the `noise()` function to generate random values for the x-coordinate of the points.
4. Yes, the code uses time-based sensory outcomes, specifically the `xoff` variable
{
    "1": False,
    "2": False,
    "3": True,
    "4": True
}

Explanation:

1. No, the code does not use visual sensory outcomes.
2. No, the code does not

Interesting, it's now explanations and hallucinating. I wonder if we force it to answer only in json it would help.

In [48]:
%%timeit
print(ask_q("""
You are a helpful assistent that only answers in valid json files.
Considering this code:
```
var xoff = 0.0
var xinc = 42

function hal() {
    let resx, resy, stepx, stepy, x, y, x1, y1, coords, p1, p2, p3, p4
    resx = 2 * Math.floor(random(2, 5)); resy = 2 * Math.floor(random(2, 5)); coords = []
    stepx = Math.floor(actualwidth / resx); stepy = Math.floor(actualheight / resy)
    x = leftmargin
    for (let i = 0; i < resx; i++) {
        y = topmargin
        for (let j = 0; j < resy; j++) {
            x1 = x + stepx * noise(xoff); xoff += xinc; y1 = y + stepy * noise(xoff)
            coords.push({ x: x1, y: y1 })
            y += stepy}
        x += stepx}
    for (let i = 0; i < resx - 1; i++) {
        for (let j = 0; j < resy - 1; j++) {
            p1 = coords[i * resy + j]; p2 = coords[i * resy + j + resy]; p3 = coords[i * resy + j + resy + 1]; p4 = coords[i * resy + j + 1]
            trianglewlines(p1.x, p1.y, p2.x, p2.y, p4.x, p4.y);
            trianglewlines(p2.x, p2.y, p3.x, p3.y, p4.x, p4.y);
}}}

function trianglewlines(x1, y1, x2, y2, x3, y3) {
    // https://mathopenref.com/coordincenter.html
    len1 = dist(x2, y2, x3, y3); len2 = dist(x1, y1, x3, y3); len3 = dist(x1, y1, x2, y2)
    cx = (len1 * x1 + len2 * x2 + len3 * x3) / (len1 + len2 + len3); cy = (len1 * y1 + len2 * y2 + len3 * y3) / (len1 + len2 + len3)
    for (let i = 0; i < 1; i += 0.03) {
        triangle(lerp(cx, x1, i), lerp(cy, y1, i), lerp(cx, x2, i), lerp(cy, y2, i), lerp(cx, x3, i), lerp(cy, y3, i),
)}}
```
Create a JSON list with boolean values with the answers for the following questions:

1) Does it uses visual sensory outcomes? [True/False]
2) Does it uses auditory sensory outcomes? [True/False]
3) Does it uses physical sensory outcomes? [True/False]
4) Does it uses static or time-based [one of these values is mandatory for this characteristic] sensory outcomes? [True/False]

Now write the valid enumerated json list file with answers and nothing else using the following template:

{
    "1": <True/False>,
    "2": <True/False>,
    "3": <True/False>,
    "4": <True/False>
}

"""))

{
    "1": False,
    "2": False,
    "3": True,
    "4": False
}
{
    "1": False,
    "2": False,
    "3": True,
    "4": False
}
{
    "1": False,
    "2": False,
    "3": True,
    "4": False
}
{
    "1": False,
    "2": False,
    "3": True,
    "4": False
}
{
    "1": False,
    "2": False,
    "3": True,
    "4": False
}
{
    "1": False,
    "2": False,
    "3": True,
    "4": False
}
{
    "1": False,
    "2": False,
    "3": True,
    "4": False
}
{
    "1": False,
    "2": False,
    "3": True,
    "4": False
}
870 ms ± 7.69 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


What if we pass all the question and the template?

In [44]:
%%timeit
print(ask_q("""
You are a helpful assistent that only answers in valid json files.
Considering this code:
```
var xoff = 0.0
var xinc = 42

function hal() {
    let resx, resy, stepx, stepy, x, y, x1, y1, coords, p1, p2, p3, p4
    resx = 2 * Math.floor(random(2, 5)); resy = 2 * Math.floor(random(2, 5)); coords = []
    stepx = Math.floor(actualwidth / resx); stepy = Math.floor(actualheight / resy)
    x = leftmargin
    for (let i = 0; i < resx; i++) {
        y = topmargin
        for (let j = 0; j < resy; j++) {
            x1 = x + stepx * noise(xoff); xoff += xinc; y1 = y + stepy * noise(xoff)
            coords.push({ x: x1, y: y1 })
            y += stepy}
        x += stepx}
    for (let i = 0; i < resx - 1; i++) {
        for (let j = 0; j < resy - 1; j++) {
            p1 = coords[i * resy + j]; p2 = coords[i * resy + j + resy]; p3 = coords[i * resy + j + resy + 1]; p4 = coords[i * resy + j + 1]
            trianglewlines(p1.x, p1.y, p2.x, p2.y, p4.x, p4.y);
            trianglewlines(p2.x, p2.y, p3.x, p3.y, p4.x, p4.y);
}}}

function trianglewlines(x1, y1, x2, y2, x3, y3) {
    // https://mathopenref.com/coordincenter.html
    len1 = dist(x2, y2, x3, y3); len2 = dist(x1, y1, x3, y3); len3 = dist(x1, y1, x2, y2)
    cx = (len1 * x1 + len2 * x2 + len3 * x3) / (len1 + len2 + len3); cy = (len1 * y1 + len2 * y2 + len3 * y3) / (len1 + len2 + len3)
    for (let i = 0; i < 1; i += 0.03) {
        triangle(lerp(cx, x1, i), lerp(cy, y1, i), lerp(cx, x2, i), lerp(cy, y2, i), lerp(cx, x3, i), lerp(cy, y3, i),
)}}
```
Create a JSON list with boolean values with the answers for the following questions:

1) Does it utilise external audio file? [True/False]
2) Does it utilise external image file? [True/False]
3) Does it synthesizes sound? [True/False]
4) Does it synthesizes images? [True/False]
5) Does it contain randomness? [True/False]
6) Does it contain interactions? [True/False]
7) Does it have human interaction (for example through mouse, midi controller, microphone, keyboard, camera, motion sensor, lidar, etc.)? [True/False]
8) Does it have computer interaction (for example through data file, data stream, remote data, web API, etc.)? [True/False]
9) Does it uses visual sensory outcomes? [True/False]
10) Does it uses auditory sensory outcomes? [True/False]
11) Does it uses physical sensory outcomes? [True/False]
12) Does it uses static or time-based [one of these values is mandatory for this characteristic] sensory outcomes? [True/False]

Now write the enumerated json list object with answers and nothing else with the following template:

{
    "1": <True/False>,
    "2": <True/False>,
    "3": <True/False>,
    "4": <True/False>,
    "5": <True/False>,
    "6": <True/False>,
    "7": <True/False>,
    "8": <True/False>,
    "9": <True/False>,
    "10": <True/False>,
    "11": <True/False>,
    "12": <True/False>
}

"""))

{
    "1": False,
    "2": False,
    "3": False,
    "4": False,
    "5": True,
    "6": False,
    "7": False,
    "8": False,
    "9": False,
    "10": False,
    "11": False,
    "12": False
}
{
    "1": False,
    "2": False,
    "3": False,
    "4": False,
    "5": True,
    "6": False,
    "7": False,
    "8": False,
    "9": False,
    "10": False,
    "11": False,
    "12": False
}
{
    "1": False,
    "2": False,
    "3": False,
    "4": False,
    "5": True,
    "6": False,
    "7": False,
    "8": False,
    "9": False,
    "10": False,
    "11": False,
    "12": False
}
{
    "1": False,
    "2": False,
    "3": False,
    "4": False,
    "5": True,
    "6": False,
    "7": False,
    "8": False,
    "9": False,
    "10": False,
    "11": False,
    "12": False
}
{
    "1": False,
    "2": False,
    "3": False,
    "4": False,
    "5": True,
    "6": False,
    "7": False,
    "8": False,
    "9": False,
    "10": False,
    "11": False,
    "12": False
}
{
    "1": Fals

Not good. Looses accuracy

Best strategy: some questions at a time.

In [42]:
%%timeit
print(ask_q("""
You are a helpful assistent that only answers in valid json files.
Considering this code:
```
var xoff = 0.0
var xinc = 42

function hal() {
    let resx, resy, stepx, stepy, x, y, x1, y1, coords, p1, p2, p3, p4
    resx = 2 * Math.floor(random(2, 5)); resy = 2 * Math.floor(random(2, 5)); coords = []
    stepx = Math.floor(actualwidth / resx); stepy = Math.floor(actualheight / resy)
    x = leftmargin
    for (let i = 0; i < resx; i++) {
        y = topmargin
        for (let j = 0; j < resy; j++) {
            x1 = x + stepx * noise(xoff); xoff += xinc; y1 = y + stepy * noise(xoff)
            coords.push({ x: x1, y: y1 })
            y += stepy}
        x += stepx}
    for (let i = 0; i < resx - 1; i++) {
        for (let j = 0; j < resy - 1; j++) {
            p1 = coords[i * resy + j]; p2 = coords[i * resy + j + resy]; p3 = coords[i * resy + j + resy + 1]; p4 = coords[i * resy + j + 1]
            trianglewlines(p1.x, p1.y, p2.x, p2.y, p4.x, p4.y);
            trianglewlines(p2.x, p2.y, p3.x, p3.y, p4.x, p4.y);
}}}

function trianglewlines(x1, y1, x2, y2, x3, y3) {
    // https://mathopenref.com/coordincenter.html
    len1 = dist(x2, y2, x3, y3); len2 = dist(x1, y1, x3, y3); len3 = dist(x1, y1, x2, y2)
    cx = (len1 * x1 + len2 * x2 + len3 * x3) / (len1 + len2 + len3); cy = (len1 * y1 + len2 * y2 + len3 * y3) / (len1 + len2 + len3)
    for (let i = 0; i < 1; i += 0.03) {
        triangle(lerp(cx, x1, i), lerp(cy, y1, i), lerp(cx, x2, i), lerp(cy, y2, i), lerp(cx, x3, i), lerp(cy, y3, i),
)}}
```
Create a JSON list with boolean values with the answers for the following questions:

1) Does it produce audio file? [True/False]
2) Does it produce image file? [True/False]
3) Does it synthesizes sound? [True/False]
4) Does it synthesizes image? [True/False]
5) Does it contain randomness? [True/False]
6) Does it contain interactions? [True/False]

Now write the enumerated json list object with answers and nothing else with the following template:

{
    "1": <True/False>,
    "2": <True/False>,
    "3": <True/False>,
    "4": <True/False>,
    "5": <True/False>,
    "6": <True/False>
}
"""))

{
    "1": False,
    "2": False,
    "3": False,
    "4": False,
    "5": True,
    "6": False
}
{
    "1": False,
    "2": False,
    "3": False,
    "4": False,
    "5": True,
    "6": False
}
{
    "1": False,
    "2": False,
    "3": False,
    "4": False,
    "5": True,
    "6": False
}
{
    "1": False,
    "2": False,
    "3": False,
    "4": False,
    "5": True,
    "6": False
}
{
    "1": False,
    "2": False,
    "3": False,
    "4": False,
    "5": True,
    "6": False
}
{
    "1": False,
    "2": False,
    "3": False,
    "4": False,
    "5": True,
    "6": False
}
{
    "1": False,
    "2": False,
    "3": False,
    "4": False,
    "5": True,
    "6": False
}
{
    "1": False,
    "2": False,
    "3": False,
    "4": False,
    "5": True,
    "6": False
}
1.23 s ± 4.57 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [51]:
%%timeit
print(ask_q("""
You are a helpful assistent that only answers in valid json files.
Considering this code:
```
var xoff = 0.0
var xinc = 42

function hal() {
    let resx, resy, stepx, stepy, x, y, x1, y1, coords, p1, p2, p3, p4
    resx = 2 * Math.floor(random(2, 5)); resy = 2 * Math.floor(random(2, 5)); coords = []
    stepx = Math.floor(actualwidth / resx); stepy = Math.floor(actualheight / resy)
    x = leftmargin
    for (let i = 0; i < resx; i++) {
        y = topmargin
        for (let j = 0; j < resy; j++) {
            x1 = x + stepx * noise(xoff); xoff += xinc; y1 = y + stepy * noise(xoff)
            coords.push({ x: x1, y: y1 })
            y += stepy}
        x += stepx}
    for (let i = 0; i < resx - 1; i++) {
        for (let j = 0; j < resy - 1; j++) {
            p1 = coords[i * resy + j]; p2 = coords[i * resy + j + resy]; p3 = coords[i * resy + j + resy + 1]; p4 = coords[i * resy + j + 1]
            trianglewlines(p1.x, p1.y, p2.x, p2.y, p4.x, p4.y);
            trianglewlines(p2.x, p2.y, p3.x, p3.y, p4.x, p4.y);
}}}

function trianglewlines(x1, y1, x2, y2, x3, y3) {
    // https://mathopenref.com/coordincenter.html
    len1 = dist(x2, y2, x3, y3); len2 = dist(x1, y1, x3, y3); len3 = dist(x1, y1, x2, y2)
    cx = (len1 * x1 + len2 * x2 + len3 * x3) / (len1 + len2 + len3); cy = (len1 * y1 + len2 * y2 + len3 * y3) / (len1 + len2 + len3)
    for (let i = 0; i < 1; i += 0.03) {
        triangle(lerp(cx, x1, i), lerp(cy, y1, i), lerp(cx, x2, i), lerp(cy, y2, i), lerp(cx, x3, i), lerp(cy, y3, i),
)}}
```
Create a JSON list with boolean values with the answers for the following questions:

1) Does it have human interaction (through mouse, midi controller, microphone, keyboard, camera, motion sensor, lidar)? [True/False]
2) Does it have computer interaction (through data file, data stream, remote data, web API)? [True/False]
Now write the valid enumerated json list file with answers and nothing else using the following template:

{
    "1": <True/False>,
    "2": <True/False>
}
"""))

{
    "1": false,
    "2": true
}
{
    "1": false,
    "2": true
}
{
    "1": false,
    "2": true
}
{
    "1": false,
    "2": true
}
{
    "1": false,
    "2": true
}
{
    "1": false,
    "2": true
}
{
    "1": false,
    "2": true
}
{
    "1": false,
    "2": true
}
510 ms ± 996 μs per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [50]:
%%timeit
print(ask_q("""
You are a helpful assistent that only answers in valid json files.
Considering this code:
```
var xoff = 0.0
var xinc = 42

function hal() {
    let resx, resy, stepx, stepy, x, y, x1, y1, coords, p1, p2, p3, p4
    resx = 2 * Math.floor(random(2, 5)); resy = 2 * Math.floor(random(2, 5)); coords = []
    stepx = Math.floor(actualwidth / resx); stepy = Math.floor(actualheight / resy)
    x = leftmargin
    for (let i = 0; i < resx; i++) {
        y = topmargin
        for (let j = 0; j < resy; j++) {
            x1 = x + stepx * noise(xoff); xoff += xinc; y1 = y + stepy * noise(xoff)
            coords.push({ x: x1, y: y1 })
            y += stepy}
        x += stepx}
    for (let i = 0; i < resx - 1; i++) {
        for (let j = 0; j < resy - 1; j++) {
            p1 = coords[i * resy + j]; p2 = coords[i * resy + j + resy]; p3 = coords[i * resy + j + resy + 1]; p4 = coords[i * resy + j + 1]
            trianglewlines(p1.x, p1.y, p2.x, p2.y, p4.x, p4.y);
            trianglewlines(p2.x, p2.y, p3.x, p3.y, p4.x, p4.y);
}}}

function trianglewlines(x1, y1, x2, y2, x3, y3) {
    // https://mathopenref.com/coordincenter.html
    len1 = dist(x2, y2, x3, y3); len2 = dist(x1, y1, x3, y3); len3 = dist(x1, y1, x2, y2)
    cx = (len1 * x1 + len2 * x2 + len3 * x3) / (len1 + len2 + len3); cy = (len1 * y1 + len2 * y2 + len3 * y3) / (len1 + len2 + len3)
    for (let i = 0; i < 1; i += 0.03) {
        triangle(lerp(cx, x1, i), lerp(cy, y1, i), lerp(cx, x2, i), lerp(cy, y2, i), lerp(cx, x3, i), lerp(cy, y3, i),
)}}
```
Create a JSON list with boolean values with the answers for the following questions:

1) Does it produce visual sensory outcomes? [True/False]
2) Does it produce auditory sensory outcomes? [True/False]
3) Does it produce physical sensory outcomes? [True/False]
4) Does it produce static or time-based [one of these values is mandatory for this characteristic] sensory outcomes? [True/False]

Now write the valid enumerated json list file with answers and nothing else using the following template:

{
    "1": <True/False>,
    "2": <True/False>,
    "3": <True/False>,
    "4": <True/False>
}
"""))

{
    "1": True,
    "2": False,
    "3": False,
    "4": True
}
{
    "1": True,
    "2": False,
    "3": False,
    "4": True
}
{
    "1": True,
    "2": False,
    "3": False,
    "4": True
}
{
    "1": True,
    "2": False,
    "3": False,
    "4": True
}
{
    "1": True,
    "2": False,
    "3": False,
    "4": True
}
{
    "1": True,
    "2": False,
    "3": False,
    "4": True
}
{
    "1": True,
    "2": False,
    "3": False,
    "4": True
}
{
    "1": True,
    "2": False,
    "3": False,
    "4": True
}
871 ms ± 4.74 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)
